# 05 — HAR volatility models (Dutta et al., 2025)

Replication of *"Impact of news and social media sentiments on rare earth investments"*
(Dutta, Sihvonen, Park, Lucey, Uddin — *Resources Policy* 107, 2025) applied to our REMX data.

**Heterogeneous Autoregressive (HAR) model** of Corsi (2009), eq. (1):

$$RV_{t,t+h} = \theta_0 + \theta_d\,RV_t + \theta_w\,RV_{t-5,t} + \theta_m\,RV_{t-22,t} + \varepsilon_t$$

- Horizons $h \in \{1, 5, 22\}$ = daily / weekly / monthly forecast horizons.
- Target $RV_{t,t+h} = \tfrac{1}{h}\sum_{j=1}^{h} RV_{t+j}$ (average future RV), eq. (2).
- Components: $RV_t$ (daily), $RV_{t-5,t}$ = 5-day trailing mean (weekly), $RV_{t-22,t}$ = 22-day trailing mean (monthly).

**Extended models** add a sentiment term in first differences (eqs. 4–7):

$$+\;\alpha\,\Delta\text{Buzz}_t \qquad\text{or}\qquad +\;\gamma\,\Delta\text{Sentiment}_t$$

**Data mapping (our case):**
| Paper | Our column |
|---|---|
| RV — Parkinson (eq. 3) | `RV_park` |
| RV — Rogers–Satchell (eq. 10) | `RV_rs` |
| Sentiment | `tone_mean` |
| Buzz | `log_article_count` |

Note: the paper separates News vs. Social media (TRMI). Our GDELT-based data is a single
source, so we estimate one `Buzz` and one `Sentiment` series (no News/Social split).


In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from pathlib import Path

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline


In [3]:
IN_PATH = 'CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = (pd.read_excel(IN_PATH, parse_dates=['Date'])
        .sort_values('Date')
        .reset_index(drop=True)
        .set_index('Date'))

print('Obs:', len(df), '| range:', df.index.min().date(), '->', df.index.max().date())
df[['RV_park', 'RV_rs', 'tone_mean', 'log_article_count']].describe()

Obs: 2766 | range: 2015-04-01 -> 2026-03-31


,RV_park,RV_rs,tone_mean,log_article_count
count,2766.000000,2766.000000,2766.000000,2766.000000
mean,0.000206,0.000205,-0.119836,4.383271
std,0.000305,0.000318,1.093931,0.695122
min,0.000003,0.000000,-7.182494,0.000000
25%,0.000062,0.000061,-0.684457,3.988984
50%,0.000117,0.000119,0.030763,4.369448
75%,0.000234,0.000232,0.606085,4.779123
max,0.004768,0.005679,3.448546,7.627544


## HAR feature construction

For each RV measure we build the three volatility components and the h-step-ahead target.
`Sentiment` (`tone_mean`) and `Buzz` (`log_article_count`) enter as first differences (Δ),
contemporaneous with the information set at $t$ (no look-ahead, since the target starts at $t+1$).


In [6]:
SENT_COL = 'tone_mean'          # paper: "Sentiment"
BUZZ_COL = 'log_article_count'  # paper: "Buzz"

RV_MEASURES = {'Parkinson': 'RV_park', 'Rogers-Satchell': 'RV_rs'}
HORIZONS    = {'Daily (h=1)': 1, 'Weekly (h=5)': 5, 'Monthly (h=22)': 22}
MODELS      = {'HAR-RV': [], 'HAR-RV-Buzz': ['dBuzz'], 'HAR-RV-Sentiment': ['dSent']}
SPLIT       = '2025-04-01'       # last 12 months held out (≈ paper's 1-year OOS)

def build_har(df, rv_col, h):
    """HAR components + Δsentiment regressors + h-step-ahead averaged RV target."""
    rv = df[rv_col]
    d = pd.DataFrame(index=df.index)
    d['RV_d'] = rv                                  # daily   : RV_t
    d['RV_w'] = rv.rolling(5).mean()                # weekly  : mean RV_{t-4..t}
    d['RV_m'] = rv.rolling(22).mean()               # monthly : mean RV_{t-21..t}
    d['dBuzz'] = df[BUZZ_COL].diff()                # Δ Buzz
    d['dSent'] = df[SENT_COL].diff()                # Δ Sentiment
    d['target'] = rv.rolling(h).mean().shift(-h)    # mean RV_{t+1..t+h}
    return d

def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

def fmt_cell(coef, se, p):
    return f"{coef:.2e}{stars(p):<3} ({se:.1e})".ljust(26)

def nw_var_mean(x, lag):
    """Newey-West HAC variance of the sample mean (Bartlett kernel)."""
    x = np.asarray(x, float); x = x - x.mean(); n = len(x)
    v = np.sum(x * x) / n
    for l in range(1, lag + 1):
        w = 1.0 - l / (lag + 1.0)
        v += 2.0 * w * np.sum(x[l:] * x[:-l]) / n
    return v / n

def fit_har(data, extra, hac_lags):
    """OLS with Newey-West (HAC) standard errors; returns res, BP p-value, DW, N."""
    cols = ['RV_d', 'RV_w', 'RV_m'] + extra
    sub = data[cols + ['target']].dropna()
    res = sm.OLS(sub['target'], sm.add_constant(sub[cols])).fit(
        cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    bp_p = sm.stats.diagnostic.het_breuschpagan(res.resid, res.model.exog)[1]
    dw   = sm.stats.stattools.durbin_watson(res.resid)
    return res, bp_p, dw, len(sub)


## In-sample HAR estimates  (paper Tables 2–7)

Full-sample estimates of the daily, weekly and monthly HAR models for both RV measures.
Standard errors are Newey-West (HAC) with lag = h, which is the standard correction for the
serial correlation induced by the overlapping (averaged) multi-horizon target — this is also
why the Durbin-Watson statistic is far below 2 for the weekly/monthly horizons (an expected
artifact of overlapping observations, not a misspecification).


In [8]:
%%capture cap_har_insample
PARAMS = ['const', 'RV_d', 'RV_w', 'RV_m', 'dBuzz', 'dSent']

for rv_name, rv_col in RV_MEASURES.items():
    print('=' * 86)
    print(f'IN-SAMPLE HAR  |  RV = {rv_name}  ({rv_col})')
    print('=' * 86)
    for hname, h in HORIZONS.items():
        data = build_har(df, rv_col, h)
        rows = {m: fit_har(data, extra, hac_lags=max(h, 1)) for m, extra in MODELS.items()}
        print(f'\n--- {hname} ---')
        print(f"{'param':<8}" + ''.join(f"{m:<26}" for m in MODELS))
        for p in PARAMS:
            line = f"{p:<8}"
            for m in MODELS:
                res = rows[m][0]
                line += fmt_cell(res.params[p], res.bse[p], res.pvalues[p]) if p in res.params.index else ' ' * 26
            print(line)
        print(f"{'R2(%)':<8}" + ''.join(f"{rows[m][0].rsquared * 100:<26.2f}" for m in MODELS))
        print(f"{'HET p':<8}" + ''.join(f"{rows[m][1]:<26.3f}" for m in MODELS))
        print(f"{'DW':<8}"    + ''.join(f"{rows[m][2]:<26.3f}" for m in MODELS))
        print(f"{'N':<8}"     + ''.join(f"{rows[m][3]:<26d}"   for m in MODELS))
    print()


In [9]:
REPORTS_DIR = Path('REPORTS'); REPORTS_DIR.mkdir(exist_ok=True)
(REPORTS_DIR / '05-HAR-models-insample.txt').write_text(cap_har_insample.stdout)
print(cap_har_insample.stdout)
print('Persisted -> REPORTS/05-HAR-models-insample.txt')


IN-SAMPLE HAR  |  RV = Parkinson  (RV_park)

--- Daily (h=1) ---
param   HAR-RV                    HAR-RV-Buzz               HAR-RV-Sentiment          
const   5.09e-05*** (9.3e-06)     5.09e-05*** (9.3e-06)     5.09e-05*** (9.4e-06)     
RV_d    3.12e-01*** (8.1e-02)     3.13e-01*** (8.1e-02)     3.12e-01*** (8.1e-02)     
RV_w    3.43e-01*** (1.0e-01)     3.43e-01*** (1.0e-01)     3.43e-01*** (1.0e-01)     
RV_m    1.01e-01    (6.7e-02)     1.01e-01    (6.7e-02)     1.01e-01    (6.7e-02)     
dBuzz                             -8.57e-06    (6.4e-06)                              
dSent                                                       1.49e-06    (3.7e-06)     
R2(%)   31.21                     31.25                     31.21                     
HET p   0.000                     0.000                     0.000                     
DW      2.052                     2.052                     2.052                     
N       2744                      2744                      2744 

## Out-of-sample forecast evaluation  (paper Tables 8–9)

In-sample estimation period: start → 2025-03-31. Out-of-sample: last 12 months (from `SPLIT`).
Parameters are estimated once on the in-sample window (the last *h* in-sample rows are dropped
so their target window does not cross into the OOS period), then forecasts are produced over OOS.

- **HRMSE** (Bollerslev–Ghysels, eq. 8): $\sqrt{\tfrac1T\sum\big((RV_t-\widehat{RV}_t)/RV_t\big)^2}$
  (observations with $RV_t=0$ are masked; the statistic is sensitive to very small $RV_t$).
- **DM** — Diebold–Mariano test (squared-error loss, HAC variance, lag $h-1$) of each extended
  model vs. the baseline HAR-RV; positive ⇒ extended model more accurate.
- **R²_OOS** (Campbell–Thompson, eq. 9): $1-\sum(RV-\widehat{RV}_{model})^2/\sum(RV-\widehat{RV}_{base})^2$;
  positive ⇒ beats baseline.
- **CW** — Clark–West MSPE-adjusted statistic (one-sided, HAC) for nested-model comparison.


In [ ]:
def oos_forecast(df, rv_col, h, extra):
    """Estimate on in-sample (< SPLIT), forecast over OOS (>= SPLIT). Returns (y, yhat)."""
    data = build_har(df, rv_col, h)
    cols = ['RV_d', 'RV_w', 'RV_m'] + extra
    is_data = data.loc[data.index < SPLIT, cols + ['target']].dropna()
    is_data = is_data.iloc[:-h] if h > 0 else is_data        # avoid target leakage across split
    beta = sm.OLS(is_data['target'], sm.add_constant(is_data[cols])).fit().params
    oos_data = data.loc[data.index >= SPLIT, cols + ['target']].dropna()
    Xo = sm.add_constant(oos_data[cols], has_constant='add')[beta.index]
    return oos_data['target'].values, Xo.values @ beta.values

def hrmse(y, yhat):
    m = y != 0
    return np.sqrt(np.mean(((y[m] - yhat[m]) / y[m]) ** 2))

def dm_test(y, f_base, f_model, lag):
    d = (y - f_base) ** 2 - (y - f_model) ** 2
    dm = d.mean() / np.sqrt(nw_var_mean(d, lag))
    return dm, 2 * (1 - stats.norm.cdf(abs(dm)))

def r2_oos(y, f_model, f_base):
    return 1 - np.sum((y - f_model) ** 2) / np.sum((y - f_base) ** 2)

def clark_west(y, f_base, f_model, lag):
    f = (y - f_base) ** 2 - ((y - f_model) ** 2 - (f_base - f_model) ** 2)
    t = f.mean() / np.sqrt(nw_var_mean(f, lag))
    return t, 1 - stats.norm.cdf(t)


In [ ]:
%%capture cap_har_oos
for rv_name, rv_col in RV_MEASURES.items():
    print('=' * 78)
    print(f'OUT-OF-SAMPLE  |  RV = {rv_name}  (split at {SPLIT})')
    print('=' * 78)
    for hname, h in HORIZONS.items():
        preds = {m: oos_forecast(df, rv_col, h, extra) for m, extra in MODELS.items()}
        yb, fb = preds['HAR-RV']
        print(f"\n--- {hname}  (OOS obs: {len(yb)}) ---")
        for m in MODELS:
            y, f = preds[m]
            if m == 'HAR-RV':
                print(f"  {m:<18} HRMSE={hrmse(y, f):.6e}   (benchmark)")
            else:
                dm, dmp = dm_test(yb, fb, f, lag=h - 1)
                cw, cwp = clark_west(y, fb, f, lag=h - 1)
                print(f"  {m:<18} HRMSE={hrmse(y, f):.6e}   "
                      f"DM={dm:+.2f}{stars(dmp):<3} R2oos={r2_oos(y, f, fb) * 100:+.4f}%   "
                      f"CW={cw:+.2f}{stars(cwp)}")
    print()


In [ ]:
(REPORTS_DIR / '05-HAR-models-oos.txt').write_text(cap_har_oos.stdout)
print(cap_har_oos.stdout)
print('Persisted -> REPORTS/05-HAR-models-oos.txt')
